# Параметрический анализ динамики модели Daisyworld при изменении светимости

В данном скрипте выполняется систематическое исследование влияния
различных параметров на поведение модели Daisyworld при изменении
солнечной светимости (сценарий :ramp). Для каждого набора параметров
строится комплексный график, включающий:
- динамику численности чёрных и белых маргариток
- изменение средней температуры поверхности
- изменение солнечной светимости во времени

## Инициализация проекта и загрузка пакетов

In [ ]:
using DrWatson
@quickactivate "project"

using Agents
using DataFrames
using Plots
using CairoMakie

### Подключение модели

Импортируем определение модели Daisyworld из исходного файла.

In [ ]:
include(srcdir("daisyworld.jl"))

## Определение агрегатных функций

Для сбора статистики о популяции маргариток определим две функции,
которые проверяют принадлежность агента к определённому виду:
- `black(a)` — возвращает `true`, если маргаритка чёрная
- `white(a)` — возвращает `true`, если маргаритка белая

In [ ]:
black(a) = a.breed == :black
white(a) = a.breed == :white

### Агрегатные данные об агентах

`adata` определяет, какие данные о агентах будут собираться в процессе
моделирования. Здесь мы собираем количество чёрных и белых маргариток
на каждом шаге.

In [ ]:
adata = [(black, count), (white, count)]

## Определение параметров эксперимента

### Структура параметров

Для исследования задаётся словарь параметров, где некоторые параметры
представлены в виде векторов. Это позволяет автоматически генерировать
все возможные комбинации значений.

**Исследуемые параметры:**
- `max_age` — максимальный возраст маргариток (25 и 40)
- `init_white` — начальная доля белых маргариток (0.2 и 0.8)

**Фиксированные параметры:**
- `griddims` — размер сетки (30×30)
- `init_black` — начальная доля чёрных маргариток (0.2)
- `albedo_white` — альбедо белых маргариток (0.75)
- `albedo_black` — альбедо чёрных маргариток (0.25)
- `surface_albedo` — альбедо почвы (0.4)
- `solar_change` — скорость изменения светимости (0.005)
- `solar_luminosity` — начальная светимость (1.0)
- `scenario` — сценарий изменения светимости (:ramp)
- `seed` — начальное значение для генератора случайных чисел (165)

**Важно:** Сценарий `:ramp` предполагает изменение светимости:
- Увеличение на 0.005 с шага 200 по 400
- Уменьшение на 0.0025 с шага 500 по 750

In [ ]:
param_dict = Dict(
    :griddims => (30, 30),
    :max_age => [25, 40],
    :init_white => [0.2, 0.8],
    :init_black => 0.2,
    :albedo_white => 0.75,
    :albedo_black => 0.25,
    :surface_albedo => 0.4,
    :solar_change => 0.005,
    :solar_luminosity => 1.0,
    :scenario => :ramp,
    :seed => 165,
)

## Генерация комбинаций параметров

Функция `dict_list` из пакета DrWatson создаёт все возможные комбинации
параметров из заданного словаря. Для каждого параметра, представленного
вектором, генерируются отдельные эксперименты.

In [ ]:
params_list = dict_list(param_dict)

## Цикл по всем комбинациям параметров

Для каждого набора параметров выполняется:
1. Создание модели с заданными параметрами
2. Определение функции для вычисления средней температуры
3. Запуск симуляции на 1000 шагов сбора данных
4. Построение комплексного графика (численность, температура, светимость)
5. Сохранение графика с уникальным именем

In [ ]:
for params in params_list

### Создание модели

Модель инициализируется с текущим набором параметров.
Используется синтаксис `;params...` для распаковки словаря
в именованные аргументы.

**Особенность:** Параметр `scenario = :ramp` активирует динамическое
изменение солнечной светимости в процессе моделирования.

In [ ]:
    model = daisyworld(; params...)

### Функция для расчёта средней температуры

Определяем функцию, которая вычисляет среднюю температуру
по всем клеткам модели. Это позволяет отслеживать глобальные
климатические изменения в ответ на изменение светимости.

In [ ]:
    temperature(model) = StatsBase.mean(model.temperature)

### Агрегатные данные о модели

`mdata` определяет, какие данные о модели будут собираться:
- средняя температура поверхности (функция temperature)
- текущая солнечная светимость (свойство модели)

In [ ]:
    mdata = [temperature, :solar_luminosity]

### Запуск моделирования

Запускаем симуляцию на 1000 шагов. Результаты сохраняются в два DataFrame:
- `agent_df` — данные об агентах (количество чёрных и белых маргариток)
- `model_df` — данные о модели (температура и светимость)

In [ ]:
    agent_df, model_df = run!(model, 1000; adata = adata, mdata = mdata)

## Построение комплексного графика

### Создание фигуры

Создаём фигуру размером 600×600 пикселей с тремя вертикальными осями.

In [ ]:
    figure = CairoMakie.Figure(size = (600, 600))

### Верхний график: численность маргариток

На первом графике отображается динамика численности маргариток:
- чёрные маргаритки — красная линия
- белые маргаритки — синяя линия

In [ ]:
    ax1 = figure[1, 1] = Axis(figure, ylabel = "daisy count")

    blackl = lines!(ax1,
        agent_df[!, :time],
        agent_df[!, :count_black],
        color = :red
    )

    whitel = lines!(ax1,
        agent_df[!, :time],
        agent_df[!, :count_white],
        color = :blue
    )

#### Легенда

Добавляем легенду для идентификации линий на верхнем графике.

In [ ]:
    figure[1, 2] = Legend(figure, [blackl, whitel], ["black", "white"])

### Средний график: температура

На втором графике отображается изменение средней температуры
поверхности во времени. Это ключевой показатель реакции системы
на изменение внешнего воздействия.

In [ ]:
    ax2 = figure[2, 1] = Axis(figure, ylabel = "temperature")

    lines!(ax2,
        model_df[!, :time],
        model_df[!, :temperature],
        color = :red
    )

### Нижний график: солнечная светимость

На третьем графике отображается изменение солнечной светимости
в соответствии со сценарием :ramp.

**Наблюдаемые фазы изменения светимости:**
- Шаги 0-200: стабильная светимость (1.0)
- Шаги 200-400: линейный рост до 2.0
- Шаги 400-500: стабильная светимость (2.0)
- Шаги 500-750: линейное падение до 0.875
- Шаги 750-1000: стабильная светимость (0.875)

In [ ]:
    ax3 = figure[3, 1] = Axis(figure,
        xlabel = "tick",
        ylabel = "luminosity"
    )

    lines!(ax3,
        model_df[!, :time],
        model_df[!, :solar_luminosity],
        color = :red
    )

### Скрытие меток времени на верхних графиках

Для улучшения читаемости комплексного графика скрываем метки времени
на верхних двух графиках. Метки отображаются только на нижнем графике,
что позволяет избежать избыточности и делает визуализацию более аккуратной.

In [ ]:
    for ax in (ax1, ax2)
        ax.xticklabelsvisible = false
    end

### Формирование имени файла

Используем функцию `savename` из пакета DrWatson для автоматического
формирования уникального имени файла на основе параметров эксперимента.
Это обеспечивает воспроизводимость и удобство идентификации результатов.

In [ ]:
    plt_name = savename("daisy-luminosity", params) * ".png"

### Сохранение результата

Сохраняем полученный комплексный график в каталог `plots/` с соответствующим именем.

In [ ]:
    save(plotsdir(plt_name), figure)
end

## Интерпретация результатов

После выполнения скрипта в каталоге `plots/` появятся комплексные графики
для каждой комбинации параметров. Анализ этих графиков позволяет:

### 1. Влияние максимального возраста (`max_age`)

- **При `max_age = 25`**:
  - Популяция более динамична, быстрее реагирует на изменение светимости
  - Колебания численности более выражены
  - Переходные процессы короче

- **При `max_age = 40`**:
  - Популяция более инертна, медленнее реагирует на изменения
  - Колебания численности сглажены
  - Переходные процессы длительнее

### 2. Влияние начального соотношения видов (`init_white`)

- **При `init_white = 0.2`** (преобладают чёрные маргаритки):
  - На начальном этапе температура выше
  - При увеличении светимости белые маргаритки начинают доминировать
  - При уменьшении светимости чёрные маргаритки восстанавливают позиции

- **При `init_white = 0.8`** (преобладают белые маргаритки):
  - На начальном этапе температура ниже
  - При увеличении светимости белые маргаритки сохраняют преимущество
  - При уменьшении светимости чёрные маргаритки получают преимущество

### 3. Анализ саморегуляции системы

**Ключевое наблюдение:** Несмотря на значительные изменения
солнечной светимости (от 1.0 до 2.0 и обратно до 0.875),
средняя температура поверхности колеблется в значительно
более узком диапазоне. Это демонстрирует способность
биосферы к саморегуляции:

- **Фаза повышения светимости (шаги 200-400)**:
  - Белые маргаритки размножаются активнее
  - Увеличение альбедо компенсирует рост светимости
  - Температура растёт медленнее, чем светимость

- **Фаза понижения светимости (шаги 500-750)**:
  - Чёрные маргаритки размножаются активнее
  - Уменьшение альбедо компенсирует падение светимости
  - Температура падает медленнее, чем светимость

### 4. Гистерезис и устойчивость

На графиках можно наблюдать явление гистерезиса —
пути изменения численности при увеличении и уменьшении
светимости не совпадают. Это указывает на наличие
нескольких устойчивых состояний системы и её способность
"запоминать" историю воздействий.